# 84 — Analyzer và báo cáo

Bậc 2 của Track 8 bắt đầu bằng câu hỏi: **chạy xong rồi, đo thế nào cho đúng?**
backtrader có sẵn cả chục analyzer, nhưng con số chúng trả về phụ thuộc vào
những tham số mặc định mà ít ai đọc. Notebook này:

| Phần | Nội dung |
|---|---|
| 1–2 | Hai feed (cổ phiếu + **VNINDEX làm benchmark**) và **lãi suất phi rủi ro lấy từ `client.macro`** |
| 3–4 | `TimeReturn`, `DrawDown`, `SharpeRatio`, `TradeAnalyzer`, `SQN`, `Transactions` → **pandas**, kèm đối chiếu chéo giữa các analyzer |
| 5–7 | Đường vốn, sụt giảm, phân phối lãi/lỗ từng vòng → **plotly** |
| 8 | ⚠️ **Cạm bẫy**: cùng một đường vốn, chín cấu hình `SharpeRatio` cho bảy con số khác nhau, từ âm tới dương |

Các mảnh dựng ở `81`–`83` (feed quy về VND, phí + thuế bán, lô 100, T+2, trần/sàn)
được import từ `finlens_examples/bt_vn.py` thay vì viết lại.

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import math

import backtrader as bt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, duong, hom_nay, lui_ngay, thanh_doi_mau
from finlens_examples import bt_vn
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


## 1 · Hai feed: cổ phiếu để giao dịch, VNINDEX để so

Benchmark nạp **thành feed thứ hai trong cùng `Cerebro`** chứ không tính riêng
bằng pandas: khi đó `TimeReturn(data=...)` đo nó trên đúng lịch phiên mà chiến
lược đi qua. Chiến lược không giao dịch feed này — nó chỉ để đo.

Hai feed có hai đơn vị khác nhau, và `bt_vn.feed_tu_finlens` đọc chúng từ
`df.attrs`: giá cổ phiếu `kVND` nhân 1.000 thành VND, còn `index_point` giữ nguyên.

In [2]:
MA = "HPG"
SO_NAM = 5
VON_DAU = 100_000_000  # VND

gia = bt_vn.bo_phien_chua_dong(client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, nam=SO_NAM)), client)
vni = bt_vn.bo_phien_chua_dong(client.eod.index.ohlcv("VNINDEX", start=lui_ngay(HOM_NAY, nam=SO_NAM)), client)

for ten, d in [(MA, gia), ("VNINDEX", vni)]:
    m = d.attrs["finlens"]
    print(
        f"{ten:<8} {len(d):>5} phiên · {d['date'].min():%d/%m/%Y} → {d['date'].max():%d/%m/%Y} · "
        f"đơn vị giá: {m['units']['close']:<11} · cơ sở giá: {m['price_basis']}"
    )

lech = set(gia["date"]) ^ set(vni["date"])
print(f"\nSố ngày có ở feed này mà không có ở feed kia: {len(lech)}")

HPG       1245 phiên · 20/10/2021 → 19/10/2026 · đơn vị giá: kVND        · cơ sở giá: adjusted
VNINDEX   1245 phiên · 20/10/2021 → 19/10/2026 · đơn vị giá: index_point · cơ sở giá: raw

Số ngày có ở feed này mà không có ở feed kia: 0


Hai lịch phiên khớp nhau hoàn toàn. Đây là điều kiện để so đường vốn với
benchmark: nếu một feed thiếu phiên, backtrader vẫn chạy, nhưng hai đường vốn
được đo trên hai lịch khác nhau mà không ai báo.

## 2 · Lãi suất phi rủi ro — lấy từ `client.macro`, không dùng số mặc định

`bt.analyzers.SharpeRatio` mặc định `riskfreerate=0.01` — 1%/năm. Lãi suất ở Việt
Nam chưa bao giờ thấp như vậy trong năm năm qua. Tìm chuỗi thật trong ba chủ đề
lãi suất của `macro`:

In [3]:
danh_muc = client.macro.indicators()
CHU_DE = ["loi_suat_trai_phieu_chinh_phu", "lai_suat_lien_ngan_hang", "lai_suat_huy_dong"]

ung_vien = danh_muc[danh_muc["topic"].isin(CHU_DE)].sort_values(["topic", "name"])
print(ung_vien[["topic", "name", "frequency", "unit", "last_date"]].to_string(index=False))

                        topic                                           name frequency   unit  last_date
            lai_suat_huy_dong                         LSHĐ_USD - Cá nhân (M)   monthly      % 2026-08-31
            lai_suat_huy_dong                         LSHĐ_USD - Tổ chức (M)   monthly      % 2026-08-31
            lai_suat_huy_dong   LSHĐ_VND - Kỳ hạn 1 - 6 tháng - Cao nhất (M)   monthly      % 2026-08-31
            lai_suat_huy_dong  LSHĐ_VND - Kỳ hạn 1 - 6 tháng - Thấp nhất (M)   monthly      % 2026-08-31
            lai_suat_huy_dong  LSHĐ_VND - Kỳ hạn 6 - 12 tháng - Cao nhất (M)   monthly      % 2026-08-31
            lai_suat_huy_dong LSHĐ_VND - Kỳ hạn 6 - 12 tháng - Thấp nhất (M)   monthly      % 2026-08-31
            lai_suat_huy_dong     LSHĐ_VND - Kỳ hạn < 1 tháng - Cao nhất (M)   monthly      % 2026-08-31
            lai_suat_huy_dong    LSHĐ_VND - Kỳ hạn < 1 tháng - Thấp nhất (M)   monthly      % 2026-08-31
            lai_suat_huy_dong    LSHĐ_VND - Kỳ hạn > 12

Chọn **lợi suất trái phiếu chính phủ kỳ hạn 1 năm**, vì:

- **Phi rủi ro theo đúng nghĩa**: nợ của Chính phủ bằng VND. Lãi suất huy động
  mang rủi ro của từng ngân hàng; liên ngân hàng là giá vốn giữa các tổ chức tín dụng.
- **Kỳ hạn khớp với cách đọc Sharpe**: Sharpe năm hoá so với lợi suất một năm.
  Lãi suất qua đêm là giá của thanh khoản trong ngày, giật mạnh mỗi khi hệ thống
  căng tiền (xem biểu đồ bên dưới).
- **Tần suất ngày**, lịch sử dài hơn cửa sổ backtest.

Chuỗi qua đêm được tải kèm chỉ để so độ biến động.

In [4]:
def tim_ma(ten: str, *, tan_suat: str = "daily") -> str:
    """Đúng MỘT mã chỉ tiêu khớp tên — không thấy hoặc mơ hồ thì dừng."""
    d = danh_muc[(danh_muc["name"] == ten) & (danh_muc["frequency"] == tan_suat)]
    if len(d) != 1:
        raise LookupError(f"{len(d)} chuỗi khớp {ten!r} ({tan_suat})")
    return str(d["code"].iloc[0])


MA_RF = tim_ma("Lợi suất trái phiếu chính phủ 1 năm")
MA_QUA_DEM = tim_ma("Lãi suất bình quân liên ngân hàng qua đêm")

lai_suat = client.macro.series([MA_RF, MA_QUA_DEM], start=lui_ngay(HOM_NAY, nam=SO_NAM + 0.1))

# ⚠️ Ở macro, `unit` là một CỘT — đọc theo từng dòng, không đọc ở df.attrs.
HE_SO_DON_VI = {"%": 0.01}  # %/năm → tỷ lệ/năm


def ve_ty_le(df: pd.DataFrame) -> pd.Series:
    la = set(df["unit"].dropna().unique()) - set(HE_SO_DON_VI)
    if la:
        raise ValueError(f"Chưa biết quy đổi đơn vị {la} — dừng thay vì đoán")
    return df["value"] * df["unit"].map(HE_SO_DON_VI).astype("float64")


lai_suat = lai_suat.assign(ty_le=ve_ty_le(lai_suat))
print("Đơn vị theo từng chuỗi:", lai_suat.groupby("code", observed=True)["unit"].unique().to_dict())
print("df.attrs nói gì về `value`?", lai_suat.attrs["finlens"]["units"]["value"])

Đơn vị theo từng chuỗi: {'lai_suat_binh_quan_lien_ngan_hang_qua_dem_daily': <StringArray>
['%']
Length: 1, dtype: string, 'loi_suat_trai_phieu_chinh_phu_1_nam_daily': <StringArray>
['%']
Length: 1, dtype: string}
df.attrs nói gì về `value`? None


In [5]:
PHIEN = pd.DatetimeIndex(gia["date"]).sort_values()


def theo_phien(code: str) -> pd.Series:
    """Chuỗi lãi suất (tỷ lệ/năm) dời về lịch phiên giao dịch: giá trị gần nhất đã biết."""
    s = lai_suat[lai_suat["code"] == code].set_index("date")["ty_le"].sort_index()
    s = s[~s.index.duplicated(keep="last")]
    return s.reindex(s.index.union(PHIEN)).ffill().reindex(PHIEN)


rf_nam = theo_phien(MA_RF)
qua_dem = theo_phien(MA_QUA_DEM)
assert rf_nam.notna().all(), "Có phiên chưa có lợi suất — mở rộng cửa sổ tải"

RF_TB = float(rf_nam.mean())
for ten, s in [("TPCP 1 năm", rf_nam), ("Qua đêm liên NH", qua_dem)]:
    print(
        f"{ten:<16} trung bình {s.mean():.2%} · thấp nhất {s.min():.2%} · cao nhất {s.max():.2%} · "
        f"gần nhất {s.iloc[-1]:.2%} · độ lệch chuẩn thay đổi ngày {s.diff().std() * 1e4:.1f} bp"
    )
print(f"\n→ RF_TB = {RF_TB:.4%}/năm — gấp {RF_TB / 0.01:.1f} lần mặc định 1% của backtrader")

TPCP 1 năm       trung bình 2.54% · thấp nhất 0.50% · cao nhất 5.03% · gần nhất 3.69% · độ lệch chuẩn thay đổi ngày 4.4 bp
Qua đêm liên NH  trung bình 3.47% · thấp nhất 0.12% · cao nhất 16.39% · gần nhất 3.58% · độ lệch chuẩn thay đổi ngày 86.3 bp

→ RF_TB = 2.5375%/năm — gấp 2.5 lần mặc định 1% của backtrader


In [6]:
ve_ls = pd.concat(
    [
        pd.DataFrame({"date": PHIEN, "ty_le": rf_nam.values * 100, "chuoi": "Lợi suất TPCP 1 năm"}),
        pd.DataFrame({"date": PHIEN, "ty_le": qua_dem.values * 100, "chuoi": "Liên ngân hàng qua đêm"}),
    ]
)
duong(
    ve_ls,
    x="date",
    y="ty_le",
    theo="chuoi",
    tieu_de="Hai ứng viên cho lãi suất phi rủi ro",
    phu_de="client.macro.series · dời về lịch phiên bằng giá trị gần nhất đã biết",
    nhan_y="%/năm",
)

## 3 · Chiến lược và bộ analyzer

Chiến lược cố ý đơn giản — giao cắt SMA 10/30 trên một mã — vì thứ cần đo ở đây
là **bộ đo**. Khung ngắn 10/30 được chọn để có đủ số vòng giao dịch cho phần phân
phối lãi/lỗ, không phải vì nó cho kết quả tốt.

Hai chi tiết đáng chú ý trong lớp dưới đây:

- `notify_order` gọi `super()` — bỏ dòng đó thì sổ T+2 của `ChienLuocVN` rỗng và
  `ban()` không bao giờ bán.
- Điều kiện bán là **trạng thái** (`nhanh < cham`) chứ không phải **sự kiện**
  giao cắt: nếu giao cắt xuống rơi vào phiên hàng chưa về (T+2), sự kiện trôi qua
  và vị thế bị kẹt tới lần giao cắt sau.

In [7]:
class GiaoCatSMA(bt_vn.ChienLuocVN):
    params = (("nhanh", 10), ("cham", 30))

    def __init__(self):
        super().__init__()  # sổ T+2
        self.sma_nhanh = bt.ind.SMA(self.data0.close, period=self.p.nhanh)
        self.sma_cham = bt.ind.SMA(self.data0.close, period=self.p.cham)
        self.cat = bt.ind.CrossOver(self.sma_nhanh, self.sma_cham)
        self.lenh = None
        self.vong = []  # nhật ký từng vòng đã đóng

    def notify_order(self, order):
        super().notify_order(order)  # bắt buộc — ghi/trừ lô T+2
        if order.status in (order.Completed, order.Canceled, order.Margin, order.Rejected, order.Expired):
            self.lenh = None

    def notify_trade(self, trade):
        if trade.isclosed:
            self.vong.append(
                {
                    "mo": bt.num2date(trade.dtopen),
                    "dong": bt.num2date(trade.dtclose),
                    "so_phien": trade.barlen,
                    "lai_gop": trade.pnl,
                    "lai_rong": trade.pnlcomm,
                }
            )

    def next(self):
        if self.lenh:  # còn lệnh chờ khớp — không đặt chồng
            return
        if not self.getposition(self.data0).size:
            if self.cat[0] > 0:
                self.lenh = self.buy(data=self.data0)
        elif self.sma_nhanh[0] < self.sma_cham[0]:
            self.lenh = self.ban(self.data0)

Gắn analyzer. `bt_vn.them_analyzer` gắn ba cái chuẩn (`loi_suat` = `TimeReturn`
theo ngày, `giao_dich` = `Transactions`, `vong` = `TradeAnalyzer`); phần còn lại
gắn tay. **Chín** `SharpeRatio` với chín bộ tham số được gắn vào **cùng một lần
chạy** — tức là đo cùng một đường vốn, đến từng đồng.

In [8]:
TF = bt.TimeFrame
CAC_SHARPE = {
    "A · mặc định": (bt.analyzers.SharpeRatio, {}),
    "B · SharpeRatio_A mặc định": (bt.analyzers.SharpeRatio_A, {}),
    "C · ngày, rf 1%": (bt.analyzers.SharpeRatio, dict(timeframe=TF.Days, riskfreerate=0.01)),
    "D · ngày, rf 1%, năm hoá": (bt.analyzers.SharpeRatio, dict(timeframe=TF.Days, riskfreerate=0.01, annualize=True)),
    "E · ngày, rf 0, năm hoá": (bt.analyzers.SharpeRatio, dict(timeframe=TF.Days, riskfreerate=0.0, annualize=True)),
    "F · ngày, rf TPCP, năm hoá": (bt.analyzers.SharpeRatio, dict(timeframe=TF.Days, riskfreerate=RF_TB, annualize=True)),
    "G · ngày, rf TPCP, năm hoá, convertrate=False": (
        bt.analyzers.SharpeRatio,
        dict(timeframe=TF.Days, riskfreerate=RF_TB, annualize=True, convertrate=False),
    ),
    "H · tháng, rf TPCP, năm hoá": (bt.analyzers.SharpeRatio, dict(timeframe=TF.Months, riskfreerate=RF_TB, annualize=True)),
    "I · ngày, rf TPCP, năm hoá, độ lệch mẫu": (
        bt.analyzers.SharpeRatio,
        dict(timeframe=TF.Days, riskfreerate=RF_TB, annualize=True, stddev_sample=True),
    ),
}


def dung_cerebro() -> bt.Cerebro:
    ce = bt.Cerebro(stdstats=False)
    ce.adddata(bt_vn.feed_tu_finlens(gia))  # data0 — giao dịch
    ce.adddata(bt_vn.feed_tu_finlens(vni))  # data1 — chỉ để đo
    ce.broker.setcash(VON_DAU)
    ce.broker.addcommissioninfo(bt_vn.PhiCoPhieuVN())
    ce.broker.set_filler(bt_vn.KhopTheoBienDo("HOSE"))
    ce.addsizer(bt_vn.LoVN)
    ce.addstrategy(GiaoCatSMA)

    bt_vn.them_analyzer(ce)  # loi_suat · giao_dich · vong
    ce.addanalyzer(bt.analyzers.DrawDown, _name="sut")
    ce.addanalyzer(bt.analyzers.SQN, _name="sqn")
    ce.addanalyzer(bt.analyzers.TimeReturn, _name="chuan", data=ce.datas[1], timeframe=TF.Days)
    ce.addanalyzer(bt.analyzers.TimeReturn, _name="theo_nam", timeframe=TF.Years)
    for i, (lop, tham_so) in enumerate(CAC_SHARPE.values()):
        ce.addanalyzer(lop, _name=f"sharpe_{i}", **tham_so)
    return ce


import time

t0 = time.perf_counter()
strat = dung_cerebro().run()[0]
print(f"Chạy xong trong {time.perf_counter() - t0:.1f} giây · {len(strat.analyzers)} analyzer trên một lần chạy")

Chạy xong trong 0.6 giây · 16 analyzer trên một lần chạy


## 4 · Analyzer → pandas

`get_analysis()` trả về `dict` lồng nhau (`AutoOrderedDict`) với khoá là
`datetime` hoặc chuỗi. `bt_vn.ket_qua` đọc ba analyzer chuẩn ra pandas:

In [9]:
kq = bt_vn.ket_qua(strat, rf_nam=rf_nam)  # rf: Series TPCP 1 năm theo phiên

print(f"loi_suat : Series {len(kq['loi_suat'])} phiên, {kq['loi_suat'].index.min():%d/%m/%Y} → {kq['loi_suat'].index.max():%d/%m/%Y}")
print(f"giao_dich: {len(kq['giao_dich'])} lần khớp\n")

NHAN = {
    "cagr": ("CAGR", "{:+.2%}"),
    "mdd": ("Sụt sâu nhất", "{:.2%}"),
    "sharpe": ("Sharpe (rf = TPCP 1 năm từng phiên)", "{:.3f}"),
    "bien_dong": ("Biến động năm hoá", "{:.2%}"),
    "so_vong_dong": ("Số vòng đã đóng", "{:d}"),
    "ty_le_thang": ("Tỷ lệ vòng thắng", "{:.1%}"),
    "so_lenh_khop": ("Số lần khớp", "{:d}"),
}
pd.DataFrame(
    [{"chỉ số": NHAN[k][0], "giá trị": NHAN[k][1].format(v)} for k, v in kq["tom_tat"].items()]
)

loi_suat : Series 1245 phiên, 20/10/2021 → 19/10/2026
giao_dich: 50 lần khớp



,chỉ số,giá trị
0,CAGR,+0.63%
1,Sụt sâu nhất,-26.99%
2,Sharpe (rf = TPCP 1 năm từng phiên),-0.001
3,Biến động năm hoá,19.26%
4,Số vòng đã đóng,25
5,Tỷ lệ vòng thắng,32.0%
6,Số lần khớp,50


In [10]:
kq["giao_dich"].head(6)

,date,symbol,so_cp,gia,gia_tri
0,2022-02-17,HPG,4000,23550.0,-94200000.0
1,2022-03-23,HPG,-4000,24160.0,96640000.0
2,2022-07-25,HPG,6400,15010.0,-96064000.0
3,2022-08-01,HPG,-6400,14680.0,93952000.0
4,2022-08-08,HPG,6000,15830.0,-94980000.0
5,2022-09-09,HPG,-6000,15790.0,94740000.0


### `TradeAnalyzer` — làm phẳng cây kết quả

Cây của `TradeAnalyzer` sâu tới bốn tầng (`won.pnl.average`…). Làm phẳng thành
một `Series` khoá chấm là cách dễ đọc nhất:

In [11]:
def phang(d: dict, tien_to: str = "") -> dict:
    ra = {}
    for k, v in d.items():
        khoa = f"{tien_to}{k}"
        ra.update(phang(v, khoa + ".") if isinstance(v, dict) else {khoa: v})
    return ra


ta = pd.Series(phang(strat.analyzers.vong.get_analysis()))
CAN_XEM = [
    "total.closed", "total.open", "won.total", "lost.total",
    "pnl.gross.total", "pnl.net.total", "pnl.net.average",
    "won.pnl.average", "lost.pnl.average", "won.pnl.max", "lost.pnl.max",
    "streak.won.longest", "streak.lost.longest",
    "len.average", "len.won.average", "len.lost.average",
]
ta.reindex(CAN_XEM).to_frame("TradeAnalyzer")

,TradeAnalyzer
total.closed,2.500000e+01
total.open,0.000000e+00
won.total,8.000000e+00
lost.total,1.700000e+01
pnl.gross.total,1.347700e+07
pnl.net.total,3.139724e+06
pnl.net.average,1.255889e+05
won.pnl.average,9.561371e+06
lost.pnl.average,-4.314779e+06
won.pnl.max,3.503105e+07


### Đối chiếu chéo: bốn nguồn phải kể cùng một câu chuyện

`TradeAnalyzer`, `Transactions`, `SQN` và nhật ký `notify_trade` đến từ bốn chỗ
khác nhau trong backtrader. Nếu chúng không khớp, con số nào trong báo cáo cũng
đáng ngờ.

In [12]:
vong = pd.DataFrame(strat.vong)
gd = kq["giao_dich"].sort_values("date").reset_index(drop=True)
so_dong = int(ta["total.closed"])
gd_dong = gd.iloc[: 2 * so_dong]  # mỗi vòng = một lần mua + một lần bán; phần dư là vị thế còn mở

phi_ghi_nhan = (vong["lai_gop"] - vong["lai_rong"]).sum()
phi_tu_tinh = (gd_dong["gia_tri"].abs() * 0.0015).sum() + (gd_dong.loc[gd_dong["so_cp"] < 0, "gia_tri"].abs() * 0.001).sum()

sqn = strat.analyzers.sqn.get_analysis()
sqn_ddof0 = math.sqrt(len(vong)) * vong["lai_rong"].mean() / vong["lai_rong"].std(ddof=0)
sqn_ddof1 = math.sqrt(len(vong)) * vong["lai_rong"].mean() / vong["lai_rong"].std(ddof=1)

print(f"Số vòng      TradeAnalyzer {so_dong} · notify_trade {len(vong)} · SQN {sqn['trades']}")
print(f"Lãi gộp      TradeAnalyzer {ta['pnl.gross.total']:>16,.0f} · Transactions {gd_dong['gia_tri'].sum():>16,.0f} VND")
print(f"Lãi ròng     TradeAnalyzer {ta['pnl.net.total']:>16,.0f} · notify_trade {vong['lai_rong'].sum():>16,.0f} VND")
print(f"Phí + thuế   ghi nhận      {phi_ghi_nhan:>16,.0f} · tự tính 0,15% + 0,1% bán {phi_tu_tinh:>11,.0f} VND")
print(f"SQN          analyzer {sqn['sqn']:.4f} · tự tính ddof=0 {sqn_ddof0:.4f} · ddof=1 {sqn_ddof1:.4f}")

Số vòng      TradeAnalyzer 25 · notify_trade 25 · SQN 25
Lãi gộp      TradeAnalyzer       13,477,000 · Transactions       13,477,000 VND
Lãi ròng     TradeAnalyzer        3,139,724 · notify_trade        3,139,724 VND
Phí + thuế   ghi nhận            10,337,276 · tự tính 0,15% + 0,1% bán  10,337,276 VND
SQN          analyzer 0.0681 · tự tính ddof=0 0.0681 · ddof=1 0.0667


Bốn nguồn khớp tới từng đồng. Hai điều rút ra:

- **`Transactions` không chứa phí** — cột `gia_tri` là `−số cp × giá` (mua âm,
  bán dương), nên cộng lại ra lãi **gộp**. Phí và thuế chỉ hiện ở chênh lệch
  `pnl` − `pnlcomm` của `TradeAnalyzer`/`notify_trade` — ở đây hơn 10 triệu, tức
  **hơn ba lần** lãi ròng cả kỳ.
- **`SQN` dùng độ lệch chuẩn tổng thể** (`ddof=0`). Với 25 vòng, chênh so với
  `ddof=1` chỉ ~2%, nhưng hệ số `√(n/(n−1))` lớn dần khi số vòng ít — ghi rõ khi
  đem SQN đi so.

## 5 · `DrawDown` — kiểm bằng cách tự tính

In [13]:
sut = strat.analyzers.sut.get_analysis()

von = kq["duong_von"] * VON_DAU  # giá trị tài khoản, VND
dinh = von.cummax()
mdd_tu_tinh = (von / dinh - 1).min()
tien_sut_tu_tinh = (dinh - von).max()

duoi_dinh = (von < dinh).astype(int)
chuoi_dai_nhat = int(duoi_dinh.groupby((duoi_dinh == 0).cumsum()).sum().max())

print(f"Sụt sâu nhất (%)   DrawDown {sut['max']['drawdown']:.6f} · tự tính {-mdd_tu_tinh * 100:.6f} "
      f"· lệch {abs(sut['max']['drawdown'] + mdd_tu_tinh * 100):.1e} điểm %")
print(f"Sụt sâu nhất (VND) DrawDown {sut['max']['moneydown']:,.0f} · tự tính {tien_sut_tu_tinh:,.0f}")
print(f"Dưới đỉnh lâu nhất DrawDown {sut['max']['len']} phiên · tự tính {chuoi_dai_nhat} phiên")

Sụt sâu nhất (%)   DrawDown 26.990460 · tự tính 26.990460 · lệch 0.0e+00 điểm %
Sụt sâu nhất (VND) DrawDown 38,048,332 · tự tính 38,048,332
Dưới đỉnh lâu nhất DrawDown 486 phiên · tự tính 486 phiên


`DrawDown` khớp với phép tính tay trên `TimeReturn` — độ sâu, số tiền, và độ dài
quãng dưới đỉnh. Lưu ý đơn vị: analyzer trả **phần trăm dương** (`26.99` nghĩa là
−26,99%), còn `bt_vn.chi_so_hieu_qua` trả
**tỷ lệ âm** (`-0.2706`). Trộn hai quy ước trong một bảng là lỗi dấu kinh điển.

## 6 · So với mua-giữ VNINDEX

`TimeReturn(data=VNINDEX)` đo lợi suất của chính feed benchmark. Đối chiếu với
`pct_change()` trên frame finlens:

In [14]:
ls_chuan = pd.Series(strat.analyzers.chuan.get_analysis(), dtype="float64")
ls_chuan.index = pd.DatetimeIndex(ls_chuan.index)
ls_pandas = vni.set_index("date")["close"].pct_change().reindex(ls_chuan.index)

print(f"Phiên đầu: TimeReturn {ls_chuan.iloc[0]:+.4%} · pct_change {ls_pandas.iloc[0]}")
print(f"  → TimeReturn lấy GIÁ MỞ CỬA phiên đầu làm gốc (firstopen=True); pandas không có phiên trước để so.")
print(f"Từ phiên thứ hai: lệch lớn nhất {(ls_chuan.iloc[1:] - ls_pandas.iloc[1:]).abs().max():.1e}")

Phiên đầu: TimeReturn -0.3325% · pct_change nan
  → TimeReturn lấy GIÁ MỞ CỬA phiên đầu làm gốc (firstopen=True); pandas không có phiên trước để so.
Từ phiên thứ hai: lệch lớn nhất 0.0e+00


In [15]:
ls_cl = kq["loi_suat"]
ls_hpg = gia.set_index("date")["close"].pct_change().reindex(ls_cl.index).fillna(0.0)
# bt_vn nhận rf là một số HOẶC Series theo ngày (phân số/năm), quy về phiên bằng lãi kép.
# Kiểm bằng tay trước khi tin:
rf_phien = (1 + rf_nam.reindex(ls_cl.index)) ** (1 / 252) - 1
vuot = ls_cl - rf_phien
tu_tinh = vuot.mean() / vuot.std(ddof=1) * math.sqrt(252)
mo_dun = bt_vn.chi_so_hieu_qua(ls_cl, rf_nam=rf_nam)["sharpe"]
print(f"Sharpe, rf TPCP từng phiên: tự tính {tu_tinh:.6f} · bt_vn {mo_dun:.6f} · "
      f"ket_qua {kq['tom_tat']['sharpe']:.6f} · lệch {abs(tu_tinh - mo_dun):.1e}")

bang = []
for ten, ls in [("Giao cắt SMA 10/30", ls_cl), ("Mua-giữ VNINDEX", ls_chuan), (f"Mua-giữ {MA}", ls_hpg)]:
    cs = bt_vn.chi_so_hieu_qua(ls, rf_nam=RF_TB)
    bang.append(
        {
            "đường vốn": ten,
            "CAGR %": round(cs["cagr"] * 100, 2),
            "biến động %": round(cs["bien_dong"] * 100, 2),
            "sụt sâu nhất %": round(cs["mdd"] * 100, 2),
            "Sharpe (rf bình quân)": round(cs["sharpe"], 3),
            "Sharpe (rf từng phiên)": round(bt_vn.chi_so_hieu_qua(ls, rf_nam=rf_nam)["sharpe"], 3),
        }
    )
pd.DataFrame(bang)

Sharpe, rf TPCP từng phiên: tự tính -0.001468 · bt_vn -0.001468 · ket_qua -0.001468 · lệch 0.0e+00


,đường vốn,CAGR %,biến động %,sụt sâu nhất %,Sharpe (rf bình quân),Sharpe (rf từng phiên)
0,Giao cắt SMA 10/30,0.63,19.26,-26.99,-0.002,-0.001
1,Mua-giữ VNINDEX,3.91,19.42,-40.34,0.166,0.166
2,Mua-giữ HPG,-6.82,32.55,-72.45,-0.131,-0.131


Chiến lược thua mua-giữ VNINDEX về lợi suất (+0,63% so với +3,91%/năm) nhưng sụt
nông hơn hẳn (−26,99% so với −40,34%), và tránh được phần lớn cú sụt −72% của
chính HPG. Cột Sharpe nói thẳng điều CAGR che đi: trừ lãi suất TPCP, chiến lược
**không kiếm thêm được gì** cho rủi ro nó gánh — Sharpe ≈ 0, còn VNINDEX ≈ 0,16.

In [16]:
def khung_von(ls: pd.Series, ten: str) -> pd.DataFrame:
    return pd.DataFrame({"date": ls.index, "von": (1 + ls).cumprod().values * 100, "duong": ten})


duong(
    pd.concat([khung_von(ls_cl, "Giao cắt SMA 10/30"), khung_von(ls_chuan, "Mua-giữ VNINDEX"), khung_von(ls_hpg, f"Mua-giữ {MA}")]),
    x="date",
    y="von",
    theo="duong",
    tieu_de=f"Đường vốn — {MA}, {SO_NAM} năm, đã trừ phí 0,15% hai chiều và thuế bán 0,1%",
    phu_de="Gốc 100 · VNINDEX là chỉ số (không phí), mua-giữ cổ phiếu dùng giá điều chỉnh",
    nhan_y="giá trị (gốc = 100)",
)

In [17]:
def khung_sut(ls: pd.Series, ten: str) -> pd.DataFrame:
    v = (1 + ls).cumprod()
    return pd.DataFrame({"date": ls.index, "sut": (v / v.cummax() - 1).values * 100, "duong": ten})


duong(
    pd.concat([khung_sut(ls_cl, "Giao cắt SMA 10/30"), khung_sut(ls_chuan, "Mua-giữ VNINDEX")]),
    x="date",
    y="sut",
    theo="duong",
    tieu_de="Sụt giảm từ đỉnh",
    phu_de="Chiến lược chỉ có vị thế khoảng một nửa số phiên — sụt nông hơn, nhưng cũng lỡ nhiều nhịp tăng",
    nhan_y="%",
    moc_khong=True,
)

## 7 · Phân phối lãi/lỗ từng vòng

In [18]:
ve_vong = vong.assign(
    nhan=lambda d: d["dong"].dt.strftime("%d/%m/%y"),
    trieu=lambda d: d["lai_rong"] / 1e6,
)
thanh_doi_mau(
    ve_vong,
    x="nhan",
    y="trieu",
    tieu_de=f"Lãi/lỗ ròng từng vòng — {len(vong)} vòng đã đóng",
    phu_de="Nhãn trục là ngày đóng vòng · đã trừ phí và thuế bán",
    nhan_y="triệu VND",
    dinh_dang_nhan="{:+.1f}",
)

In [19]:
thang = vong[vong["lai_rong"] > 0]["lai_rong"]
thua = vong[vong["lai_rong"] <= 0]["lai_rong"]
print(f"Thắng {len(thang)} vòng, lãi bình quân {thang.mean() / 1e6:+.2f} triệu · giữ bình quân {vong.loc[thang.index, 'so_phien'].mean():.0f} phiên")
print(f"Thua  {len(thua)} vòng, lỗ bình quân  {thua.mean() / 1e6:+.2f} triệu · giữ bình quân {vong.loc[thua.index, 'so_phien'].mean():.0f} phiên")
print(f"Tỷ số lãi/lỗ bình quân: {thang.mean() / abs(thua.mean()):.2f} · kỳ vọng mỗi vòng {vong['lai_rong'].mean() / 1e6:+.2f} triệu")
print(f"Ba vòng lãi lớn nhất cộng lại {thang.nlargest(3).sum() / 1e6:+.1f} triệu — "
      f"gấp {thang.nlargest(3).sum() / vong['lai_rong'].sum():.0f} lần lãi ròng cả kỳ ({vong['lai_rong'].sum() / 1e6:+.1f} triệu)")
print(f"Thời gian có vị thế: {vong['so_phien'].sum() / len(kq['loi_suat']):.0%} số phiên")

Thắng 8 vòng, lãi bình quân +9.56 triệu · giữ bình quân 49 phiên
Thua  17 vòng, lỗ bình quân  -4.31 triệu · giữ bình quân 13 phiên
Tỷ số lãi/lỗ bình quân: 2.22 · kỳ vọng mỗi vòng +0.13 triệu
Ba vòng lãi lớn nhất cộng lại +63.9 triệu — gấp 20 lần lãi ròng cả kỳ (+3.1 triệu)
Thời gian có vị thế: 49% số phiên


Đây là hình dạng quen thuộc của chiến lược đi theo xu hướng: **thua nhiều vòng
nhỏ, thắng ít vòng lớn**. Tỷ lệ thắng một mình nói rất ít — phải đọc cùng tỷ số
lãi/lỗ bình quân. Ở đây ba vòng lãi lớn nhất cộng lại gấp khoảng 20 lần lãi ròng
cả kỳ: bỏ lỡ **một** trong số đó (vì trần trắng bên, vì nghỉ tay một tuần) là đủ
đổi dấu kết quả.

## 8 · ⚠️ Cạm bẫy: một đường vốn, chín cấu hình Sharpe

Chín `SharpeRatio` dưới đây đọc **cùng một chuỗi giá trị tài khoản** từ cùng một
lần chạy. Thêm ba cách tự tính để đối chiếu.

In [20]:
TEN_TF = {TF.Days: "ngày", TF.Weeks: "tuần", TF.Months: "tháng", TF.Years: "năm"}

hang = []
for i, ten in enumerate(CAC_SHARPE):
    a = strat.analyzers.getbyname(f"sharpe_{i}")
    hang.append(
        {
            "cách tính": ten,
            "timeframe": TEN_TF[a.p.timeframe],
            "riskfreerate": f"{a.p.riskfreerate:.2%}",
            "annualize": a.p.annualize,
            "convertrate": a.p.convertrate,
            "stddev_sample": a.p.stddev_sample,
            "Sharpe": a.get_analysis()["sharperatio"],
        }
    )

for ten, gia_tri, rf in [
    ("J · bt_vn.chi_so_hieu_qua, rf 0", bt_vn.chi_so_hieu_qua(ls_cl, rf_nam=0.0)["sharpe"], "0.00%"),
    ("K · bt_vn.chi_so_hieu_qua, rf TPCP bình quân", bt_vn.chi_so_hieu_qua(ls_cl, rf_nam=RF_TB)["sharpe"], f"{RF_TB:.2%}"),
    ("L · bt_vn.chi_so_hieu_qua, rf TPCP từng phiên", bt_vn.chi_so_hieu_qua(ls_cl, rf_nam=rf_nam)["sharpe"], "theo phiên"),
]:
    hang.append({"cách tính": ten, "timeframe": "ngày", "riskfreerate": rf, "annualize": True,
                 "convertrate": "—", "stddev_sample": True, "Sharpe": gia_tri})

bang_sharpe = pd.DataFrame(hang)
bang_sharpe["Sharpe"] = bang_sharpe["Sharpe"].astype(float).round(4)
print(f"Cùng một đường vốn → Sharpe từ {bang_sharpe['Sharpe'].min():.4f} tới {bang_sharpe['Sharpe'].max():.4f}")
bang_sharpe

Cùng một đường vốn → Sharpe từ -0.0258 tới 0.1285


,cách tính,timeframe,riskfreerate,annualize,convertrate,stddev_sample,Sharpe
0,A · mặc định,năm,1.00%,False,True,False,0.0409
1,B · SharpeRatio_A mặc định,năm,1.00%,True,True,False,0.0409
2,"C · ngày, rf 1%",ngày,1.00%,False,True,False,0.0048
3,"D · ngày, rf 1%, năm hoá",ngày,1.00%,True,True,False,0.0768
4,"E · ngày, rf 0, năm hoá",ngày,0.00%,True,True,False,0.1285
5,"F · ngày, rf TPCP, năm hoá",ngày,2.54%,True,True,False,-0.0017
6,"G · ngày, rf TPCP, năm hoá, convertrate=False",ngày,2.54%,True,False,False,0.0715
7,"H · tháng, rf TPCP, năm hoá",tháng,2.54%,True,True,False,-0.0258
8,"I · ngày, rf TPCP, năm hoá, độ lệch mẫu",ngày,2.54%,True,True,True,-0.0017
9,"J · bt_vn.chi_so_hieu_qua, rf 0",ngày,0.00%,True,—,True,0.1284


Vì sao mặc định (`A`) lệch hẳn khỏi các cách tính theo ngày? Nó đo trên lợi suất
**theo năm dương lịch**:

In [21]:
theo_nam = pd.Series(strat.analyzers.theo_nam.get_analysis(), dtype="float64")
so_phien_nam = kq["loi_suat"].groupby(kq["loi_suat"].index.year).size()
pd.DataFrame(
    {
        "lợi suất năm %": (theo_nam.values * 100).round(2),
        "số phiên trong năm": so_phien_nam.values,
    },
    index=[d.year for d in theo_nam.index],
)

,lợi suất năm %,số phiên trong năm
2021,0.00,53
2022,-5.02,249
2023,11.26,249
2024,-1.74,250
2025,26.84,249
2026,-21.69,195


Mỗi con số của analyzer tái hiện được bằng pandas, khi đã biết nó làm gì bên
trong (`backtrader/analyzers/sharpe.py`, hàm `stop`):

In [22]:
def sharpe_kieu_bt(ls: pd.Series, rf: float, factor: int, *, convertrate=True, annualize=False, ddof=0) -> float:
    """Chép lại đúng logic SharpeRatio.stop() của backtrader 1.9.78."""
    if convertrate:
        rate = (1 + rf) ** (1 / factor) - 1  # hạ rf năm xuống kỳ
    else:
        rate, ls = rf, (1 + ls) ** factor - 1  # "nâng" TỪNG lợi suất kỳ lên năm
    vuot = ls - rate
    ty_so = vuot.mean() / vuot.std(ddof=ddof)
    return ty_so * math.sqrt(factor) if (convertrate and annualize) else ty_so


ls_thang = (1 + ls_cl).groupby(ls_cl.index.to_period("M")).prod() - 1
TAI_HIEN = {
    "A": sharpe_kieu_bt(theo_nam, 0.01, 1),
    "C": sharpe_kieu_bt(ls_cl, 0.01, 252),
    "F": sharpe_kieu_bt(ls_cl, RF_TB, 252, annualize=True),
    "G": sharpe_kieu_bt(ls_cl, RF_TB, 252, convertrate=False, annualize=True),
    "H": sharpe_kieu_bt(ls_thang, RF_TB, 12, annualize=True),
}
for k, v in TAI_HIEN.items():
    goc = bang_sharpe.loc[bang_sharpe["cách tính"].str.startswith(k), "Sharpe"].iloc[0]
    print(f"{k}: analyzer {goc:+.4f} · tự tính {v:+.4f}")

vuot_chia_deu = ls_cl - RF_TB / 252  # quy rf kiểu "chia đều", thay vì lãi kép
print(f"K nếu quy rf bằng RF_TB/252: {vuot_chia_deu.mean() / vuot_chia_deu.std(ddof=1) * math.sqrt(252):+.4f}")

nang = (1 + ls_cl) ** 252 - 1
print(f"\nCách G 'nâng' mỗi lợi suất ngày lên năm: phiên tốt nhất {ls_cl.max():+.2%} → {nang.max():+,.0%}/năm, "
      f"phiên tệ nhất {ls_cl.min():+.2%} → {nang.min():+.2%}/năm")

A: analyzer +0.0409 · tự tính +0.0409
C: analyzer +0.0048 · tự tính +0.0048
F: analyzer -0.0017 · tự tính -0.0017
G: analyzer +0.0715 · tự tính +0.0715
H: analyzer -0.0258 · tự tính -0.0258
K nếu quy rf bằng RF_TB/252: -0.0034

Cách G 'nâng' mỗi lợi suất ngày lên năm: phiên tốt nhất +6.71% → +1,287,479,995%/năm, phiên tệ nhất -6.77% → -100.00%/năm


### Đọc bảng — cái nào đúng, và vì sao

Mốc để đọc: chiến lược có CAGR **+0,63%/năm**, còn trái phiếu chính phủ 1 năm trả
bình quân **2,54%/năm** trong cùng kỳ. Chiến lược **không thắng được tín phiếu** —
Sharpe đúng phải quanh 0, hơi âm. Đối chiếu từng dòng:

| Cách | Sharpe | Vấn đề |
|---|---|---|
| **A** mặc định | +0,041 | Sharpe của **6 điểm lợi suất năm dương lịch**, trong đó 2021 chỉ có 53 phiên (chiến lược chưa vào lệnh, lợi suất 0) và 2026 có 195 phiên — năm dở dang được tính như năm đủ. Cộng thêm rf 1% không có thật. Sáu điểm dữ liệu thì độ lệch chuẩn gần như là nhiễu. |
| **B** `SharpeRatio_A` | +0,041 | Tên gợi ý "đã năm hoá", nhưng khung mặc định là năm nên năm hoá là nhân với √1 — **trùng A**. |
| **C** ngày, không năm hoá | +0,005 | Sharpe **mỗi phiên**: phép tính đúng, nhưng nhỏ hơn √252 ≈ 15,9 lần đơn vị người ta vẫn trích dẫn. |
| **D** ngày, rf 1% | +0,077 | Đúng khung, sai lãi suất: rf thấp hơn thực tế → Sharpe **đổi dấu** thành dương. |
| **E** ngày, rf 0 | +0,129 | Lạc quan nhất. Chỉ dùng được để so nội bộ giữa các chiến lược. |
| **F** ngày, rf TPCP | −0,002 | **Đúng.** Lợi suất ngày, trừ lợi suất TPCP 1 năm bình quân kỳ (hạ về ngày bằng lãi kép), năm hoá × √252. |
| **G** `convertrate=False` | +0,072 | Không hạ rf về ngày mà **"nâng" từng lợi suất ngày lên năm** bằng `(1+r)^252 − 1`: phiên tốt nhất (+6,71%) thành hơn một tỷ phần trăm một năm (dòng in ở trên), rồi bỏ luôn bước √252. Ra một con số trông hợp lý nhưng không đo cái gì — không lỗi, không cảnh báo. |
| **H** tháng | −0,026 | Hợp lệ, nhưng chỉ ~60 điểm nên nhiễu hơn F. Lệch so với F là chuyện bình thường. |
| **I** ngày, `ddof=1` | −0,002 | Như F, đổi độ lệch tổng thể sang độ lệch mẫu — khác từ chữ số thứ năm. |
| **J, K** `bt_vn` | +0,128 / −0,002 | Cùng phép tính với backtrader: rf hạ về phiên bằng lãi kép, năm hoá × √252, độ lệch mẫu (`ddof=1`) — nên J ≈ E và **K trùng I** tới chữ số thứ tư. Quy rf bằng cách chia đều `rf/252` thì K thành −0,003: với Sharpe sát 0, chừng đó đủ để nhân đôi con số. |
| **L** rf từng phiên | −0,0015 | Đúng nhất về nguyên tắc: mỗi phiên trừ đúng lãi suất của phiên đó (TPCP dao động 0,50%–5,03% trong kỳ). |

Cùng một đường vốn, con số dao động từ **−0,026 tới +0,129** và **đổi dấu** tuỳ tham
số. Con số nên báo cáo: **F / L** (K nếu không dùng backtrader), kèm ghi rõ khung
thời gian, nguồn rf và cách năm hoá. Một Sharpe không kèm ba thông tin đó thì không
so được với bất cứ thứ gì.

`bt_vn.chi_so_hieu_qua` (và `ket_qua`) nhận `rf_nam` là một số **hoặc** một Series
theo ngày, quy về phiên bằng lãi kép như `SharpeRatio` — cách L là một dòng gọi hàm,
và phần 6 đã kiểm nó khớp với phép tính tay.

## Tổng kết

| Điều | Thực hiện thế nào |
|---|---|
| Benchmark cùng lịch phiên | VNINDEX là feed thứ hai, đo bằng `TimeReturn(data=...)` |
| rf có nguồn | `client.macro`, lợi suất TPCP 1 năm, đơn vị đọc theo **từng dòng** |
| Analyzer khớp nhau | `TradeAnalyzer` = `Transactions` = `notify_trade` = `SQN`, tới từng đồng |
| `DrawDown` đáng tin | khớp với MDD tự tính trên `TimeReturn` |
| Sharpe có ngữ cảnh | khung ngày · rf thật · năm hoá √252 — và in cả ba thông tin đó |
| Đọc phí đúng chỗ | `Transactions` là giá trị gộp; phí + thuế chỉ có trong `pnlcomm` |

---

Tiếp theo: [`85_chi_bao_va_du_lieu_ngoai_gia.ipynb`](85_chi_bao_va_du_lieu_ngoai_gia.ipynb) —
chỉ báo của backtrader so với `df.finlens`, và đưa dòng tiền khối ngoại, P/E, ROE
vào làm tín hiệu mà không nhìn trước.